# 03 · Preparação para a modelagem

**CRISP-DM fase 4 — preparação dos dados** · Meridiano Imobiliária

---

O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) entregou a **base curada**: 2.350
imóveis, 83 colunas, nenhuma célula vazia e cada ajuste com motivo registrado. Ela descreve
a carteira em **unidades de negócio**: pés quadrados, dólares, notas de 0 a 5, e é
justamente isso que a torna legível para a Meridiano e imprópria para um algoritmo baseado
em distância.

In [1]:
# ============================================================================
# Configuração de avisos
# ============================================================================
import warnings

def warn(*args, **kwargs):
    pass

warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# ============================================================================
# Importações
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
# from category_encoders import CountEncoder

# ============================================================================
# Escalonamento de variáveis numéricas
# ============================================================================
# MinMaxScaler    → ajusta cada variável (coluna) para o intervalo de 0 a 1.
#
# StandardScaler  → centraliza cada variável (coluna) em média 0 e desvio
#                   padrão 1. Não torna a distribuição normal.
#
# Normalizer      → ajusta cada observação (linha) para norma L2 igual a 1,
#                   ou seja, a soma dos quadrados dos valores da linha é 1.

# ============================================================================
# Codificação de variáveis categóricas
# ============================================================================
# OneHotEncoder   → cria uma coluna binária (0/1) para cada categoria.
#                   Indicado para variáveis nominais.
#
# OrdinalEncoder  → converte cada categoria em um número inteiro. A ordem só
#                   é respeitada se informada em `categories`; caso contrário,
#                   segue a ordem alfabética. Indicado para variáveis ordinais.
#
# CountEncoder    → substitui cada categoria pela sua frequência na base
#                   (biblioteca category_encoders).

In [2]:
# Carregando a base de dados tratada de RH e o nosso dicionário
df = pd.read_csv("../data/processed/base_rh_tratado.csv", sep=';')
dicionario = pd.read_csv("../data/processed/dicionario.csv", sep=';')

In [3]:
useless_columns = ['numero_do_funcionario'] 

In [ ]:
def colunas_do_tipo(*tipos):
    colunas = dicionario.loc[dicionario['Tipo'].isin(tipos), 'variavel']
    return [c for c in colunas if c in df.columns and c not in useless_columns]

continuous_columns = colunas_do_tipo('Contínua', 'Discreta')

ordinal_columns = colunas_do_tipo('Ordinal')
nominal_columns = colunas_do_tipo('Nominal')

derived_features = ['faixa_etaria', 'categoria_tempo_empresa', 'alta_mobilidade_carreira']
nominal_columns = list(dict.fromkeys(nominal_columns + ['faixa_etaria', 'categoria_tempo_empresa']))
continuous_columns = list(dict.fromkeys(continuous_columns + ['alta_mobilidade_carreira']))

In [6]:
if ordinal_columns != []:
    for col in ordinal_columns:
        pass
else:
    print("Nenhuma coluna ordinal na base de RH. Passando adiante.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

continuous_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) 
])

nominal_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

imatrix_transformers = [
    ('continuous', continuous_preprocessor, [c for c in continuous_columns if c in df.columns]),
    ('nominal', nominal_preprocessor, [c for c in nominal_columns if c in df.columns])
]

# Montando o ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=imatrix_transformers, 
    verbose_feature_names_out=False
).set_output(transform='pandas')

In [9]:
X = df.drop(columns=[c for c in useless_columns if c in df.columns], errors='ignore')

X_preprocessed = preprocessor.fit_transform(X)

X_preprocessed.to_csv("../data/processed/base_rh_preprocessed.csv", sep=';', index=False)
print("Base de RH pré-processada e salva com sucesso em '../data/processed/base_rh_preprocessed.csv'!")

Base de RH pré-processada e salva com sucesso em '../data/processed/base_rh_preprocessed.csv'!
